# Stage B validation — AERONET-blind held-out (§8.2.2)

The headline gap-fill quality test.

**Blind protocol:** keep only `(slot, site)` pairs where AERONET observed but the Stage A daily product was *missing* at the AERONET cell.  Only gap-fill points count — observed pixels are excluded because they're not what's being graded.

AERONET aggregation: ±30 min around slot centre, identical to the Stage A protocol (§8.0).

In [ ]:
from datetime import date
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
import validate as vb
import config as cfg

START = cfg.TEST_START
END   = cfg.TEST_END
print(f'Held-out window: {START} → {END}')

## Extract blind pairs for the RF candidate

One row per `(slot, site)` matched pair with AERONET, candidate value, provenance, uncertainty.

In [ ]:
pairs_rf = vb.aeronet_pairs(START, END, candidate='rf', blind_only=True)
print(f'AERONET-blind RF pairs: {len(pairs_rf)}')
pairs_rf.head()

## Metric panel — RF candidate

Per-site, per-(site, season), and pooled.  Metrics: `N`, `R`, `R²`, `RMSE`, `MAE`, `Bias`, `pct_EE` (within `0.05 + 0.15·|AERONET|` expected-error envelope).

In [ ]:
vb.metric_panel(pairs_rf)

## Scatter — RF vs AERONET

Visual sanity check; 1:1 line drawn for reference.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
if pairs_rf.empty:
    ax.set_title('No matched pairs')
else:
    for site, marker in zip(pairs_rf['site'].unique(), ('o', 's', '^', 'D')):
        sub = pairs_rf[pairs_rf['site'] == site]
        ax.scatter(sub['aer_aod'], sub['sat_aod'], alpha=0.6, label=site, marker=marker)
    lim = max(pairs_rf[['aer_aod', 'sat_aod']].max().max(), 1.0)
    ax.plot([0, lim], [0, lim], 'k--', lw=0.8)
    ax.set_xlabel('AERONET AOD'); ax.set_ylabel('RF gap-filled AOD')
    ax.set_title('Stage B RF — AERONET-blind held-out')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

# Krigging section

In [ ]:
pairs_k = vb.aeronet_pairs(START, END, candidate='kriging', blind_only=True)
print(f'AERONET-blind Kriging pairs: {len(pairs_k)}')
pairs_k.head()

In [ ]:
vb.metric_panel(pairs_k)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
if pairs_k.empty:
    ax.set_title('No matched pairs')
else:
    for site, marker in zip(pairs_k['site'].unique(), ('o', 's', '^', 'D')):
        sub = pairs_k[pairs_k['site'] == site]
        ax.scatter(sub['aer_aod'], sub['sat_aod'], alpha=0.6, label=site, marker=marker)
    lim = max(pairs_k[['aer_aod', 'sat_aod']].max().max(), 1.0)
    ax.plot([0, lim], [0, lim], 'k--', lw=0.8)
    ax.set_xlabel('AERONET AOD'); ax.set_ylabel('Kriging gap-filled AOD')
    ax.set_title('Stage B Kriging — AERONET-blind held-out')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()